In [0]:
from pyspark.sql import functions as F

## Create silver schema

We create the final_project.silver schema, which will store the cleaned taxi data.  
All transformations in this notebook write their results into this schema.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS final_project.silver;

In [0]:
train_bronze = spark.table("final_project.bronze.train")

In [0]:
test_bronze  = spark.table("final_project.bronze.test")

## Apply cleaning rules to train and test



In this step we apply the main data quality filters:

- Drop rows with nulls in critical columns
- Keep only reasonable fare_amount values (0–500)
- Restrict passenger_count to [1, 6]
- Filter trips to a valid NYC bounding box for pickup and dropoff coordinates

These rules remove obvious outliers and corrupted records from the raw data.

In [0]:
test_clean = (
    test_bronze
        .dropna()
        .filter((F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 6))
        .filter(
            (F.col("pickup_longitude") >= -75) &
            (F.col("pickup_longitude") <= -72) &
            (F.col("dropoff_longitude") >= -75) &
            (F.col("dropoff_longitude") <= -72) &
            (F.col("pickup_latitude") >= 40) &
            (F.col("pickup_latitude") <= 42) &
            (F.col("dropoff_latitude") >= 40) &
            (F.col("dropoff_latitude") <= 42)
        )
)

In [0]:
train_clean = (
    train_bronze
        .dropna()
        .filter((F.col("fare_amount") > 0) & (F.col("fare_amount") <= 500))
        .filter((F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 6))
        .filter(
            (F.col("pickup_longitude") >= -75) &
            (F.col("pickup_longitude") <= -72) &
            (F.col("dropoff_longitude") >= -75) &
            (F.col("dropoff_longitude") <= -72) &
            (F.col("pickup_latitude") >= 40) &
            (F.col("pickup_latitude") <= 42) &
            (F.col("dropoff_latitude") >= 40) &
            (F.col("dropoff_latitude") <= 42)
        )
)

## Standardize column types



We now enforce consistent data types across both tables, for example:

- Cast pickup_datetime to timestamp
- Cast coordinates to double
- Cast passenger_count to int
- Cast fare_amount to double (train only)

This makes the silver tables ready for feature engineering and ML pipelines.

In [0]:
train_silver = (
    train_clean
        .withColumn("fare_amount",       F.col("fare_amount").cast("double"))
        .withColumn("pickup_datetime",   F.to_timestamp("pickup_datetime"))
        .withColumn("pickup_longitude",  F.col("pickup_longitude").cast("double"))
        .withColumn("pickup_latitude",   F.col("pickup_latitude").cast("double"))
        .withColumn("dropoff_longitude", F.col("dropoff_longitude").cast("double"))
        .withColumn("dropoff_latitude",  F.col("dropoff_latitude").cast("double"))
        .withColumn("passenger_count",   F.col("passenger_count").cast("int"))
)

In [0]:
test_silver = (
    test_clean
        .withColumn("pickup_datetime",   F.to_timestamp("pickup_datetime"))
        .withColumn("pickup_longitude",  F.col("pickup_longitude").cast("double"))
        .withColumn("pickup_latitude",   F.col("pickup_latitude").cast("double"))
        .withColumn("dropoff_longitude", F.col("dropoff_longitude").cast("double"))
        .withColumn("dropoff_latitude",  F.col("dropoff_latitude").cast("double"))
        .withColumn("passenger_count",   F.col("passenger_count").cast("int"))
)

In [0]:
(
    train_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("final_project.silver.train")
)
(
    test_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("final_project.silver.test")
)
print("Silver tables were created")

In [0]:
silver_train = spark.table("final_project.silver.train")

## Sanity checks on silver tables



We validate the silver tables by:

- Checking row counts for train and test
- Counting remaining null values per column

This helps verify that the cleaning rules were applied correctly and that key columns are fully populated.


In [0]:
silver_train_rows = silver_train.count()
print(f"Rows in final_project.silver.train: {silver_train_rows}")

In [0]:
null_counts_train = silver_train.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_train.columns
])

print("Null counts per column in final_project.silver.train:")
display(null_counts_train)